# **Practical No. 7: Design and Implementation of an ETL Pipeline Using Airflow**

## **Pipeline 1 — CSV ETL**

STEP 1 — Import Libraries

In [ ]:
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


STEP 2 — Create a Sample CSV File

In [ ]:
data = {
    "Order_ID": [1, 2, 3, 4, 5],
    "Customer": ["Rahul", "Sneha", "Amit", "Priya", "Neha"],
    "Product": ["Laptop", "Mouse", "Keyboard", "Mobile", "Chair"],
    "Quantity": [1, 2, 3, 1, 2],
    "Price": [50000, 2000, 3000, 30000, 8000]
}

df = pd.DataFrame(data)

df.to_csv("sales.csv", index=False)

print("sales.csv created successfully!")

sales.csv created successfully!


STEP 3 — Extract Data from CSV

In [ ]:
df = pd.read_csv("sales.csv")

print("CSV data extracted successfully!")
display(df)

CSV data extracted successfully!


,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,3,Amit,Keyboard,3,3000
3,4,Priya,Mobile,1,30000
4,5,Neha,Chair,2,8000


STEP 4 — Check Missing Values

In [ ]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
Order_ID    0
Customer    0
Product     0
Quantity    0
Price       0
dtype: int64


STEP 5 — Remove Duplicate Records

In [ ]:
df = df.drop_duplicates()

print("Duplicate records removed!")
display(df)

Duplicate records removed!


,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,3,Amit,Keyboard,3,3000
3,4,Priya,Mobile,1,30000
4,5,Neha,Chair,2,8000


STEP 6 — Clean the Data

In [ ]:
df["Customer"] = df["Customer"].str.strip().str.upper()
df["Product"] = df["Product"].str.strip().str.upper()

print("Data cleaning completed!")
display(df)

Data cleaning completed!


,Order_ID,Customer,Product,Quantity,Price
0,1,RAHUL,LAPTOP,1,50000
1,2,SNEHA,MOUSE,2,2000
2,3,AMIT,KEYBOARD,3,3000
3,4,PRIYA,MOBILE,1,30000
4,5,NEHA,CHAIR,2,8000


STEP 7 — Validate the Data

In [ ]:
df = df[
    (df["Quantity"] > 0) &
    (df["Price"] > 0)
].copy()

print("Data validation completed!")
display(df)

Data validation completed!


,Order_ID,Customer,Product,Quantity,Price
0,1,RAHUL,LAPTOP,1,50000
1,2,SNEHA,MOUSE,2,2000
2,3,AMIT,KEYBOARD,3,3000
3,4,PRIYA,MOBILE,1,30000
4,5,NEHA,CHAIR,2,8000


STEP 8 — Transform the Data

In [ ]:
df["Total_Amount"] = df["Quantity"] * df["Price"]

print("Data transformation completed!")
display(df)

Data transformation completed!


,Order_ID,Customer,Product,Quantity,Price,Total_Amount
0,1,RAHUL,LAPTOP,1,50000,50000
1,2,SNEHA,MOUSE,2,2000,4000
2,3,AMIT,KEYBOARD,3,3000,9000
3,4,PRIYA,MOBILE,1,30000,30000
4,5,NEHA,CHAIR,2,8000,16000


STEP 9 — Create SQLite Database

In [ ]:
connection = sqlite3.connect("sales_database.db")

print("Database created successfully!")

Database created successfully!


STEP 10 — Load Data into Database

In [ ]:
df.to_sql(
    "sales",
    connection,
    if_exists="replace",
    index=False
)

print("Data loaded into database successfully!")

Data loaded into database successfully!


STEP 11 — Read Data from Database

In [ ]:
result = pd.read_sql(
    "SELECT * FROM sales",
    connection
)

display(result)

,Order_ID,Customer,Product,Quantity,Price,Total_Amount
0,1,RAHUL,LAPTOP,1,50000,50000
1,2,SNEHA,MOUSE,2,2000,4000
2,3,AMIT,KEYBOARD,3,3000,9000
3,4,PRIYA,MOBILE,1,30000,30000
4,5,NEHA,CHAIR,2,8000,16000


STEP 12 — Generate a Simple Report

In [ ]:
report = pd.read_sql(
    """
    SELECT
        Product,
        SUM(Quantity) AS Total_Quantity,
        SUM(Total_Amount) AS Total_Sales
    FROM sales
    GROUP BY Product
    """,
    connection
)

display(report)

,Product,Total_Quantity,Total_Sales
0,CHAIR,2,16000
1,KEYBOARD,3,9000
2,LAPTOP,1,50000
3,MOBILE,1,30000
4,MOUSE,2,4000


STEP 13 — Close the Database

In [ ]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


## **Pipeline 2: Multiple CSV Files**

STEP 1 — Create the First CSV File

In [ ]:
import pandas as pd

data1 = {
    "Order_ID": [101, 102, 103],
    "Customer": ["Rahul", "Sneha", "Amit"],
    "Product": ["Laptop", "Mouse", "Keyboard"],
    "Quantity": [1, 2, 3]
}

df1 = pd.DataFrame(data1)

df1.to_csv("sales_part1.csv", index=False)

print("sales_part1.csv created successfully!")

sales_part1.csv created successfully!


STEP 2 — Create the Second CSV File

In [ ]:
data2 = {
    "Order_ID": [104, 105, 106],
    "Customer": ["Priya", "Neha", "Rohan"],
    "Product": ["Mobile", "Chair", "Laptop"],
    "Quantity": [1, 2, 1]
}

df2 = pd.DataFrame(data2)

df2.to_csv("sales_part2.csv", index=False)

print("sales_part2.csv created successfully!")

sales_part2.csv created successfully!


STEP 3 — Read the First CSV

In [ ]:
sales1 = pd.read_csv("sales_part1.csv")

print("First CSV:")
display(sales1)

First CSV:


,Order_ID,Customer,Product,Quantity
0,101,Rahul,Laptop,1
1,102,Sneha,Mouse,2
2,103,Amit,Keyboard,3


STEP 4 — Read the Second CSV

In [ ]:
sales2 = pd.read_csv("sales_part2.csv")

print("Second CSV:")
display(sales2)

Second CSV:


,Order_ID,Customer,Product,Quantity
0,104,Priya,Mobile,1
1,105,Neha,Chair,2
2,106,Rohan,Laptop,1


STEP 5 — Combine Both CSV Files

In [ ]:
combined_df = pd.concat(
    [sales1, sales2],
    ignore_index=True
)

print("Combined Dataset:")
display(combined_df)

Combined Dataset:


,Order_ID,Customer,Product,Quantity
0,101,Rahul,Laptop,1
1,102,Sneha,Mouse,2
2,103,Amit,Keyboard,3
3,104,Priya,Mobile,1
4,105,Neha,Chair,2
5,106,Rohan,Laptop,1


STEP 6 — Check the Combined Dataset

In [ ]:
print("Total records:", len(combined_df))

Total records: 6


In [ ]:
print("Columns:")
print(combined_df.columns.tolist())

Columns:
['Order_ID', 'Customer', 'Product', 'Quantity']


STEP 7 — Check Missing Values

In [ ]:
print("Missing values:")
print(combined_df.isnull().sum())

Missing values:
Order_ID    0
Customer    0
Product     0
Quantity    0
dtype: int64


STEP 8 — Check Duplicate Records

In [ ]:
print(
    "Duplicate records:",
    combined_df.duplicated().sum()
)

Duplicate records: 0


STEP 9 — Save the Combined Dataset

In [ ]:
combined_df.to_csv(
    "combined_sales.csv",
    index=False
)

print("Combined dataset saved successfully!")

Combined dataset saved successfully!


STEP 10 — Verify the Final File

In [ ]:
final_df = pd.read_csv(
    "combined_sales.csv"
)

display(final_df)

,Order_ID,Customer,Product,Quantity
0,101,Rahul,Laptop,1
1,102,Sneha,Mouse,2
2,103,Amit,Keyboard,3
3,104,Priya,Mobile,1
4,105,Neha,Chair,2
5,106,Rohan,Laptop,1


## **Pipeline 3: JSON ETL**

STEP 1 — Import Libraries

In [ ]:
import json
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


STEP 2 — Create a JSON File

In [ ]:
json_data = {
    "customers": [
        {
            "id": 1,
            "name": "Rahul",
            "email": "rahul@gmail.com",
            "city": "Mumbai",
            "age": 21
        },
        {
            "id": 2,
            "name": "Sneha",
            "email": "sneha@gmail.com",
            "city": "Pune",
            "age": 20
        },
        {
            "id": 3,
            "name": "Amit",
            "email": "amit@gmail.com",
            "city": "Nashik",
            "age": 22
        },
        {
            "id": 4,
            "name": "Priya",
            "email": "priya@gmail.com",
            "city": "Thane",
            "age": 21
        }
    ]
}

with open("customers.json", "w") as file:
    json.dump(json_data, file, indent=4)

print("customers.json created successfully!")

customers.json created successfully!


STEP 3 — Extract JSON Data

In [ ]:
with open("customers.json", "r") as file:
    data = json.load(file)

print("JSON data extracted successfully!")
print(data)

JSON data extracted successfully!
{'customers': [{'id': 1, 'name': 'Rahul', 'email': 'rahul@gmail.com', 'city': 'Mumbai', 'age': 21}, {'id': 2, 'name': 'Sneha', 'email': 'sneha@gmail.com', 'city': 'Pune', 'age': 20}, {'id': 3, 'name': 'Amit', 'email': 'amit@gmail.com', 'city': 'Nashik', 'age': 22}, {'id': 4, 'name': 'Priya', 'email': 'priya@gmail.com', 'city': 'Thane', 'age': 21}]}


STEP 4 — Convert JSON into a DataFrame

In [ ]:
customers_df = pd.DataFrame(
    data["customers"]
)

display(customers_df)

,id,name,email,city,age
0,1,Rahul,rahul@gmail.com,Mumbai,21
1,2,Sneha,sneha@gmail.com,Pune,20
2,3,Amit,amit@gmail.com,Nashik,22
3,4,Priya,priya@gmail.com,Thane,21


STEP 5 — Select Required Fields

In [ ]:
selected_df = customers_df[
    ["id", "name", "city"]
].copy()

display(selected_df)

,id,name,city
0,1,Rahul,Mumbai
1,2,Sneha,Pune
2,3,Amit,Nashik
3,4,Priya,Thane


STEP 6 — Transform the Selected Fields

In [ ]:
selected_df["name"] = (
    selected_df["name"]
    .str.upper()
)

display(selected_df)

,id,name,city
0,1,RAHUL,Mumbai
1,2,SNEHA,Pune
2,3,AMIT,Nashik
3,4,PRIYA,Thane


STEP 7 — Standardize City Names

In [ ]:
selected_df["city"] = (
    selected_df["city"]
    .str.upper()
)

display(selected_df)

,id,name,city
0,1,RAHUL,MUMBAI
1,2,SNEHA,PUNE
2,3,AMIT,NASHIK
3,4,PRIYA,THANE


STEP 8 — Validate the Transformed Data

In [ ]:
print("Missing values:")
print(selected_df.isnull().sum())

Missing values:
id      0
name    0
city    0
dtype: int64


In [ ]:
print(
    "Duplicate IDs:",
    selected_df["id"].duplicated().sum()
)

Duplicate IDs: 0


STEP 9 — Create Relational Database

In [ ]:
connection = sqlite3.connect(
    "customer_database.db"
)

print("Database created successfully!")

Database created successfully!


STEP 10 — Load JSON Data into Database

In [ ]:
selected_df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("JSON data loaded into database successfully!")

JSON data loaded into database successfully!


STEP 11 — Check the Database Table

In [ ]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    connection
)

display(tables)

,name
0,customers


STEP 12 — Read Data from SQL Database

In [ ]:
result = pd.read_sql(
    "SELECT * FROM customers",
    connection
)

display(result)

,id,name,city
0,1,RAHUL,MUMBAI
1,2,SNEHA,PUNE
2,3,AMIT,NASHIK
3,4,PRIYA,THANE


STEP 13 — Run a SQL Query

In [ ]:
mumbai_customers = pd.read_sql(
    """
    SELECT *
    FROM customers
    WHERE city = 'MUMBAI'
    """,
    connection
)

display(mumbai_customers)

,id,name,city
0,1,RAHUL,MUMBAI


STEP 14 — Close Database

In [ ]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


## **Pipeline 4: Identify and Remove Invalid Records**

STEP 1 — Import Libraries

In [ ]:
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


STEP 2 — Create a Dataset with Invalid Records

In [ ]:
data = {
    "Order_ID": [1, 2, 3, 4, 5, 6],
    "Customer": ["Rahul", "Sneha", "Amit", None, "Neha", "Riya"],
    "Product": ["Laptop", "Mouse", "Keyboard", "Mobile", "Chair", "Laptop"],
    "Quantity": [1, 2, -3, 1, 0, 2],
    "Price": [50000, 2000, 3000, -30000, 8000, 50000]
}

df = pd.DataFrame(data)

print("Original Dataset:")
display(df)

Original Dataset:


,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,3,Amit,Keyboard,-3,3000
3,4,None,Mobile,1,-30000
4,5,Neha,Chair,0,8000
5,6,Riya,Laptop,2,50000


STEP 3 — Check Missing Values

In [ ]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
Order_ID    0
Customer    1
Product     0
Quantity    0
Price       0
dtype: int64


STEP 4 — Identify Records with Missing Values

In [ ]:
missing_records = df[
    df.isnull().any(axis=1)
]

print("Records with missing values:")
display(missing_records)

Records with missing values:


,Order_ID,Customer,Product,Quantity,Price
3,4,None,Mobile,1,-30000


STEP 5 — Identify Invalid Quantity

In [ ]:
invalid_quantity = df[
    df["Quantity"] <= 0
]

print("Records with invalid quantity:")
display(invalid_quantity)

Records with invalid quantity:


,Order_ID,Customer,Product,Quantity,Price
2,3,Amit,Keyboard,-3,3000
4,5,Neha,Chair,0,8000


STEP 6 — Identify Invalid Price

In [ ]:
invalid_price = df[
    df["Price"] <= 0
]

print("Records with invalid price:")
display(invalid_price)

Records with invalid price:


,Order_ID,Customer,Product,Quantity,Price
3,4,None,Mobile,1,-30000


STEP 7 — Identify All Invalid Records

In [ ]:
invalid_records = df[
    df["Customer"].isnull() |
    (df["Quantity"] <= 0) |
    (df["Price"] <= 0)
]

print("All Invalid Records:")
display(invalid_records)

All Invalid Records:


,Order_ID,Customer,Product,Quantity,Price
2,3,Amit,Keyboard,-3,3000
3,4,None,Mobile,1,-30000
4,5,Neha,Chair,0,8000


STEP 8 — Count Invalid Records

In [ ]:
print(
    "Number of invalid records:",
    len(invalid_records)
)

Number of invalid records: 3


STEP 9 — Remove Invalid Records

In [ ]:
cleaned_df = df[
    df["Customer"].notnull() &
    (df["Quantity"] > 0) &
    (df["Price"] > 0)
].copy()

print("Cleaned Dataset:")
display(cleaned_df)

Cleaned Dataset:


,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
5,6,Riya,Laptop,2,50000


STEP 10 — Verify the Cleaned Data

In [ ]:
print(
    "Invalid quantities:",
    (cleaned_df["Quantity"] <= 0).sum()
)

print(
    "Invalid prices:",
    (cleaned_df["Price"] <= 0).sum()
)

print(
    "Missing customer names:",
    cleaned_df["Customer"].isnull().sum()
)

Invalid quantities: 0
Invalid prices: 0
Missing customer names: 0


STEP 11 — Save the Clean Dataset

In [ ]:
cleaned_df.to_csv(
    "cleaned_sales.csv",
    index=False
)

print("Cleaned data saved successfully!")

Cleaned data saved successfully!


STEP 12 — Create SQLite Database

In [ ]:
connection = sqlite3.connect(
    "cleaned_sales_database.db"
)

print("Database created successfully!")

Database created successfully!


STEP 13 — Load Only Valid Records

In [ ]:
cleaned_df.to_sql(
    "valid_sales",
    connection,
    if_exists="replace",
    index=False
)

print("Valid records loaded into database!")

Valid records loaded into database!


STEP 14 — Verify Database

In [ ]:
result = pd.read_sql(
    "SELECT * FROM valid_sales",
    connection
)

display(result)

,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,6,Riya,Laptop,2,50000


STEP 15 — Check Number of Records in Database

In [ ]:
count = pd.read_sql(
    "SELECT COUNT(*) AS Total_Records FROM valid_sales",
    connection
)

display(count)

,Total_Records
0,3


STEP 16 — Close Database

In [ ]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


## **Pipeline 5: Data Validation Before Database Loading**

STEP 1 — Import Libraries

In [ ]:
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


STEP 2 — Create Sample Data

In [ ]:
data = {
    "Student_ID": [1, 2, 3, 4, 5],
    "Name": ["Rahul", "Sneha", "Amit", "Priya", "Neha"],
    "Age": [21, 20, 22, -5, 25],
    "Email": [
        "rahul@gmail.com",
        "sneha@gmail.com",
        "amit@gmail.com",
        "priya@gmail.com",
        "neha@gmail.com"
    ],
    "Marks": [85, 90, 75, 110, 65]
}

df = pd.DataFrame(data)

print("Original Data:")
display(df)

Original Data:


,Student_ID,Name,Age,Email,Marks
0,1,Rahul,21,rahul@gmail.com,85
1,2,Sneha,20,sneha@gmail.com,90
2,3,Amit,22,amit@gmail.com,75
3,4,Priya,-5,priya@gmail.com,110
4,5,Neha,25,neha@gmail.com,65


STEP 3 — Validate Student ID

In [ ]:
valid_id = df["Student_ID"].notnull()

print("Valid Student IDs:")
print(valid_id)

Valid Student IDs:
0    True
1    True
2    True
3    True
4    True
Name: Student_ID, dtype: bool


STEP 4 — Validate Names

In [ ]:
valid_name = df["Name"].notnull()

print("Valid Names:")
print(valid_name)

Valid Names:
0    True
1    True
2    True
3    True
4    True
Name: Name, dtype: bool


STEP 5 — Validate Age

In [ ]:
valid_age = df["Age"] > 0

print("Valid Ages:")
print(valid_age)

Valid Ages:
0     True
1     True
2     True
3    False
4     True
Name: Age, dtype: bool


STEP 6 — Validate Email

In [ ]:
valid_email = df["Email"].str.contains("@", na=False)

print("Valid Emails:")
print(valid_email)

Valid Emails:
0    True
1    True
2    True
3    True
4    True
Name: Email, dtype: bool


STEP 7 — Validate Marks

In [ ]:
valid_marks = (
    (df["Marks"] >= 0) &
    (df["Marks"] <= 100)
)

print("Valid Marks:")
print(valid_marks)

Valid Marks:
0     True
1     True
2     True
3    False
4     True
Name: Marks, dtype: bool


STEP 8 — Combine All Validation Rules

In [ ]:
validation_result = (
    valid_id &
    valid_name &
    valid_age &
    valid_email &
    valid_marks
)

print("Overall Validation Result:")
print(validation_result)

Overall Validation Result:
0     True
1     True
2     True
3    False
4     True
dtype: bool


STEP 9 — Separate Valid and Invalid Records

In [ ]:
valid_data = df[
    validation_result
].copy()

print("Valid Records:")
display(valid_data)

Valid Records:


,Student_ID,Name,Age,Email,Marks
0,1,Rahul,21,rahul@gmail.com,85
1,2,Sneha,20,sneha@gmail.com,90
2,3,Amit,22,amit@gmail.com,75
4,5,Neha,25,neha@gmail.com,65


In [ ]:
invalid_data = df[
    ~validation_result
].copy()

print("Invalid Records:")
display(invalid_data)

Invalid Records:


,Student_ID,Name,Age,Email,Marks
3,4,Priya,-5,priya@gmail.com,110


STEP 10 — Display Validation Summary

In [ ]:
print("Total Records:", len(df))
print("Valid Records:", len(valid_data))
print("Invalid Records:", len(invalid_data))

Total Records: 5
Valid Records: 4
Invalid Records: 1


STEP 11 — Create Target Database

In [ ]:
connection = sqlite3.connect(
    "student_validation.db"
)

print("Database created successfully!")

Database created successfully!


STEP 12 — Load ONLY Valid Data

In [ ]:
if not valid_data.empty:
    valid_data.to_sql(
        "students",
        connection,
        if_exists="replace",
        index=False
    )

    print("Valid data loaded into database successfully!")
else:
    print("No valid data available for loading.")

Valid data loaded into database successfully!


STEP 13 — Store Invalid Data Separately

In [ ]:
invalid_data.to_csv(
    "invalid_students.csv",
    index=False
)

print("Invalid records saved separately.")

Invalid records saved separately.


STEP 14 — Verify Database

In [ ]:
result = pd.read_sql(
    "SELECT * FROM students",
    connection
)

print("Data stored in database:")
display(result)

Data stored in database:


,Student_ID,Name,Age,Email,Marks
0,1,Rahul,21,rahul@gmail.com,85
1,2,Sneha,20,sneha@gmail.com,90
2,3,Amit,22,amit@gmail.com,75
3,5,Neha,25,neha@gmail.com,65


STEP 15 — Count Database Records

In [ ]:
count = pd.read_sql(
    "SELECT COUNT(*) AS Total_Students FROM students",
    connection
)

display(count)

,Total_Students
0,4


STEP 16 — Close Database

In [ ]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


## **Pipeline 6: Incremental Data Loading**

STEP 1 — Import Libraries

In [ ]:
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


STEP 2 — Create Initial Dataset

In [ ]:
initial_data = {
    "Order_ID": [1, 2, 3, 4, 5],
    "Customer": ["Rahul", "Sneha", "Amit", "Priya", "Neha"],
    "Product": ["Laptop", "Mouse", "Keyboard", "Mobile", "Chair"],
    "Quantity": [1, 2, 3, 1, 2],
    "Price": [50000, 2000, 3000, 30000, 8000]
}

existing_df = pd.DataFrame(initial_data)

display(existing_df)

,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,3,Amit,Keyboard,3,3000
3,4,Priya,Mobile,1,30000
4,5,Neha,Chair,2,8000


STEP 3 — Create the Database

In [ ]:
connection = sqlite3.connect("incremental_database.db")

print("Database created successfully!")

Database created successfully!


STEP 4 — Load Initial Data into Database

In [ ]:
existing_df.to_sql(
    "sales",
    connection,
    if_exists="replace",
    index=False
)

print("Initial data loaded into database!")

Initial data loaded into database!


STEP 5 — Check Existing Data

In [ ]:
existing_data = pd.read_sql(
    "SELECT * FROM sales",
    connection
)

display(existing_data)

,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,3,Amit,Keyboard,3,3000
3,4,Priya,Mobile,1,30000
4,5,Neha,Chair,2,8000


STEP 6 — Create Newly Arriving Data

In [ ]:
new_data = {
    "Order_ID": [6, 7, 8],
    "Customer": ["Riya", "Karan", "Pooja"],
    "Product": ["Laptop", "Mobile", "Keyboard"],
    "Quantity": [1, 2, 3],
    "Price": [50000, 30000, 3000]
}

new_df = pd.DataFrame(new_data)

new_df.to_csv(
    "new_sales.csv",
    index=False
)

print("New sales file created!")
display(new_df)

New sales file created!


,Order_ID,Customer,Product,Quantity,Price
0,6,Riya,Laptop,1,50000
1,7,Karan,Mobile,2,30000
2,8,Pooja,Keyboard,3,3000


STEP 7 — Read the New CSV

In [ ]:
new_sales = pd.read_csv(
    "new_sales.csv"
)

print("New data extracted:")
display(new_sales)

New data extracted:


,Order_ID,Customer,Product,Quantity,Price
0,6,Riya,Laptop,1,50000
1,7,Karan,Mobile,2,30000
2,8,Pooja,Keyboard,3,3000


STEP 8 — Get Existing Order IDs

In [ ]:
existing_orders = pd.read_sql(
    "SELECT Order_ID FROM sales",
    connection
)

print("Existing Order IDs:")
display(existing_orders)

Existing Order IDs:


,Order_ID
0,1
1,2
2,3
3,4
4,5


STEP 9 — Find Only New Records

In [ ]:
incremental_data = new_sales[
    ~new_sales["Order_ID"].isin(
        existing_orders["Order_ID"]
    )
].copy()

print("Only New Records:")
display(incremental_data)

Only New Records:


,Order_ID,Customer,Product,Quantity,Price
0,6,Riya,Laptop,1,50000
1,7,Karan,Mobile,2,30000
2,8,Pooja,Keyboard,3,3000


STEP 10 — Validate New Records

In [ ]:
valid_data = incremental_data[
    (incremental_data["Quantity"] > 0) &
    (incremental_data["Price"] > 0) &
    (incremental_data["Customer"].notnull())
].copy()

print("Validated New Records:")
display(valid_data)

Validated New Records:


,Order_ID,Customer,Product,Quantity,Price
0,6,Riya,Laptop,1,50000
1,7,Karan,Mobile,2,30000
2,8,Pooja,Keyboard,3,3000


STEP 11 — Load Only New Records

In [ ]:
if not valid_data.empty:
    valid_data.to_sql(
        "sales",
        connection,
        if_exists="append",
        index=False
    )

    print("New records loaded successfully!")
else:
    print("No new valid records to load.")

New records loaded successfully!


STEP 12 — Verify the Database

In [ ]:
final_data = pd.read_sql(
    "SELECT * FROM sales",
    connection
)

display(final_data)

,Order_ID,Customer,Product,Quantity,Price
0,1,Rahul,Laptop,1,50000
1,2,Sneha,Mouse,2,2000
2,3,Amit,Keyboard,3,3000
3,4,Priya,Mobile,1,30000
4,5,Neha,Chair,2,8000
5,6,Riya,Laptop,1,50000
6,7,Karan,Mobile,2,30000
7,8,Pooja,Keyboard,3,3000


STEP 13 — Check Total Records

In [ ]:
count = pd.read_sql(
    "SELECT COUNT(*) AS Total_Records FROM sales",
    connection
)

display(count)

,Total_Records
0,8


STEP 14 — Test Incremental Loading Again

In [ ]:
existing_orders = pd.read_sql(
    "SELECT Order_ID FROM sales",
    connection
)

new_records = new_sales[
    ~new_sales["Order_ID"].isin(
        existing_orders["Order_ID"]
    )
]

print("New records available:", len(new_records))
display(new_records)

New records available: 0


,Order_ID,Customer,Product,Quantity,Price


STEP 15 — Close Database

In [ ]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


In [ ]:
# PRACTICAL NO. 7
# ETL Pipeline: CSV + JSON → Clean → Transform → Validate → Incremental Load → SQLite Database

import pandas as pd
import sqlite3
import os

# ---------------------------------------------------------
# 1. CREATE SAMPLE CSV AND JSON FILES
# ---------------------------------------------------------

csv_file = "customers.csv"
json_file = "orders.json"
database_file = "etl_database.db"

# Sample CSV data
csv_data = pd.DataFrame({
    "CustomerID": [1, 2, 3, 4, 5, 6],
    "Name": ["Alice", "Bob", "Charlie", "David", "Eva", "Invalid"],
    "Email": [
        "alice@gmail.com",
        "bob@gmail.com",
        "charlie@gmail.com",
        "david@gmail.com",
        "eva@gmail.com",
        "invalid-email"
    ],
    "Age": [25, 32, 28, 40, 35, -5]
})

csv_data.to_csv(csv_file, index=False)

# Sample JSON data
json_data = [
    {"OrderID": 101, "CustomerID": 1, "Product": "Laptop", "Amount": 55000},
    {"OrderID": 102, "CustomerID": 2, "Product": "Mobile", "Amount": 25000},
    {"OrderID": 103, "CustomerID": 3, "Product": "Tablet", "Amount": 18000},
    {"OrderID": 104, "CustomerID": 4, "Product": "Headphones", "Amount": 3000},
    {"OrderID": 105, "CustomerID": 5, "Product": "Keyboard", "Amount": 2000},
    {"OrderID": 106, "CustomerID": 6, "Product": "Mouse", "Amount": -500}
]

pd.DataFrame(json_data).to_json(json_file, orient="records", indent=4)

# ---------------------------------------------------------
# 2. EXTRACT DATA FROM CSV
# ---------------------------------------------------------

customers = pd.read_csv(csv_file)

# Extract data from JSON
orders = pd.read_json(json_file)

print("CSV Data:")
print(customers)

print("\nJSON Data:")
print(orders)

# ---------------------------------------------------------
# 3. CLEAN AND TRANSFORM CUSTOMER DATA
# ---------------------------------------------------------

# Remove duplicate records
customers = customers.drop_duplicates()

# Remove extra spaces
customers["Name"] = customers["Name"].astype(str).str.strip()
customers["Email"] = customers["Email"].astype(str).str.strip().str.lower()

# Convert Age to numeric
customers["Age"] = pd.to_numeric(customers["Age"], errors="coerce")

# ---------------------------------------------------------
# 4. CLEAN AND TRANSFORM ORDER DATA
# ---------------------------------------------------------

orders = orders.drop_duplicates()

orders["Product"] = orders["Product"].astype(str).str.strip()

orders["CustomerID"] = pd.to_numeric(
    orders["CustomerID"], errors="coerce"
)

orders["Amount"] = pd.to_numeric(
    orders["Amount"], errors="coerce"
)

# ---------------------------------------------------------
# 5. IDENTIFY INVALID CUSTOMER RECORDS
# ---------------------------------------------------------

invalid_customers = customers[
    (customers["CustomerID"].isna()) |
    (customers["Name"].isna()) |
    (customers["Email"].isna()) |
    (~customers["Email"].str.contains("@", na=False)) |
    (customers["Age"].isna()) |
    (customers["Age"] <= 0)
]

# Keep only valid customers
valid_customers = customers[
    ~customers.index.isin(invalid_customers.index)
].copy()

# ---------------------------------------------------------
# 6. IDENTIFY INVALID ORDER RECORDS
# ---------------------------------------------------------

invalid_orders = orders[
    (orders["OrderID"].isna()) |
    (orders["CustomerID"].isna()) |
    (orders["Product"].isna()) |
    (orders["Amount"].isna()) |
    (orders["Amount"] <= 0)
]

# Keep only valid orders
valid_orders = orders[
    ~orders.index.isin(invalid_orders.index)
].copy()

# ---------------------------------------------------------
# 7. DATA VALIDATION
# ---------------------------------------------------------

# Validate customer IDs
valid_customer_ids = set(valid_customers["CustomerID"])

# Orders must belong to an existing customer
valid_orders = valid_orders[
    valid_orders["CustomerID"].isin(valid_customer_ids)
].copy()

# Validate positive transaction amount
valid_orders = valid_orders[
    valid_orders["Amount"] > 0
].copy()

# ---------------------------------------------------------
# 8. DISPLAY INVALID RECORDS
# ---------------------------------------------------------

print("\nInvalid Customer Records Removed:")
print(invalid_customers)

print("\nInvalid Order Records Removed:")
print(invalid_orders)

# ---------------------------------------------------------
# 9. CONNECT TO RELATIONAL DATABASE
# ---------------------------------------------------------

conn = sqlite3.connect(database_file)

# ---------------------------------------------------------
# 10. CREATE TABLES IF NOT EXISTS
# ---------------------------------------------------------

conn.execute("""
CREATE TABLE IF NOT EXISTS customers (
    CustomerID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Email TEXT NOT NULL,
    Age INTEGER NOT NULL
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS orders (
    OrderID INTEGER PRIMARY KEY,
    CustomerID INTEGER NOT NULL,
    Product TEXT NOT NULL,
    Amount REAL NOT NULL,
    FOREIGN KEY (CustomerID) REFERENCES customers(CustomerID)
)
""")

conn.commit()

# ---------------------------------------------------------
# 11. INCREMENTAL LOADING - CUSTOMERS
# ---------------------------------------------------------

existing_customers = pd.read_sql_query(
    "SELECT CustomerID FROM customers",
    conn
)

existing_customer_ids = set(
    existing_customers["CustomerID"].tolist()
)

new_customers = valid_customers[
    ~valid_customers["CustomerID"].isin(existing_customer_ids)
].copy()

if not new_customers.empty:
    new_customers.to_sql(
        "customers",
        conn,
        if_exists="append",
        index=False
    )

# ---------------------------------------------------------
# 12. INCREMENTAL LOADING - ORDERS
# ---------------------------------------------------------

existing_orders = pd.read_sql_query(
    "SELECT OrderID FROM orders",
    conn
)

existing_order_ids = set(
    existing_orders["OrderID"].tolist()
)

new_orders = valid_orders[
    ~valid_orders["OrderID"].isin(existing_order_ids)
].copy()

if not new_orders.empty:
    new_orders.to_sql(
        "orders",
        conn,
        if_exists="append",
        index=False
    )

# ---------------------------------------------------------
# 13. FINAL DATA VALIDATION IN DATABASE
# ---------------------------------------------------------

final_customers = pd.read_sql_query(
    "SELECT * FROM customers",
    conn
)

final_orders = pd.read_sql_query(
    "SELECT * FROM orders",
    conn
)

# Check for invalid database records
invalid_db_customers = final_customers[
    (final_customers["Age"] <= 0) |
    (~final_customers["Email"].str.contains("@", na=False))
]

invalid_db_orders = final_orders[
    (final_orders["Amount"] <= 0)
]

# ---------------------------------------------------------
# 14. DISPLAY FINAL RESULTS
# ---------------------------------------------------------

print("\n========== ETL PIPELINE RESULT ==========")

print("\nValid Customers Loaded:")
print(final_customers)

print("\nValid Orders Loaded:")
print(final_orders)

print("\nNumber of Invalid Customers Removed:",
      len(invalid_customers))

print("Number of Invalid Orders Removed:",
      len(invalid_orders))

print("New Customers Loaded:",
      len(new_customers))

print("New Orders Loaded:",
      len(new_orders))

print("Invalid Records in Database:",
      len(invalid_db_customers) + len(invalid_db_orders))

print("\nETL Pipeline Completed Successfully!")

# ---------------------------------------------------------
# 15. CLOSE DATABASE CONNECTION
# ---------------------------------------------------------

conn.close()

CSV Data:
   CustomerID     Name              Email  Age
0           1    Alice    alice@gmail.com   25
1           2      Bob      bob@gmail.com   32
2           3  Charlie  charlie@gmail.com   28
3           4    David    david@gmail.com   40
4           5      Eva      eva@gmail.com   35
5           6  Invalid      invalid-email   -5

JSON Data:
   OrderID  CustomerID     Product  Amount
0      101           1      Laptop   55000
1      102           2      Mobile   25000
2      103           3      Tablet   18000
3      104           4  Headphones    3000
4      105           5    Keyboard    2000
5      106           6       Mouse    -500

Invalid Customer Records Removed:
   CustomerID     Name          Email  Age
5           6  Invalid  invalid-email   -5

Invalid Order Records Removed:
   OrderID  CustomerID Product  Amount
5      106           6   Mouse    -500

========== ETL PIPELINE RESULT ==========

Valid Customers Loaded:
   CustomerID     Name              Email  Age
0 

In [ ]:
print("Mayuri Narsu Patil")

Mayuri Narsu Patil
